# cartcell LOMO on Colab GPU

Runs `lomo.py` (leave-one-allele-out evaluation) from
[Cartcell-Caspnet](https://github.com/Ryan-Bae0121/Cartcell-Caspnet) with GPU acceleration.

**Before running**: Runtime -> Change runtime type -> Hardware accelerator: GPU.

Run the cells in order. `lomo.py` resumes automatically from whatever is already in
`reports_lomo/` (skips any allele already finished), so re-running this notebook after a
disconnect just continues where it left off -- as long as Google Drive is mounted (cell 3),
since Colab wipes the local disk between sessions.

## 1. Clone the repo

In [ ]:
!git clone https://github.com/Ryan-Bae0121/Cartcell-Caspnet.git
%cd Cartcell-Caspnet/codes/custom_codes

## 2. Confirm GPU is available

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 3. Mount Google Drive (persists results across disconnects)

First run only: copies the progress already pushed to GitHub (1/49 alleles as of this
notebook's creation) into Drive. On later reconnects, skip the `cp` line -- Drive already
has the latest.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/cartcell/reports_lomo
!cp -rn reports_lomo/* /content/drive/MyDrive/cartcell/reports_lomo/ 2>/dev/null || true
%env OUT_DIR=/content/drive/MyDrive/cartcell/reports_lomo

## 4. Run LOMO

Full run (49 alleles, Phase2 30ep + Phase3 40ep each). Remove `OUT_DIR` env (cell 3) if you
don't want Drive persistence for a quick test, and use `ALLELE_LIMIT` for a small smoke test
first, e.g. `!ALLELE_LIMIT=3 python lomo.py`.

In [ ]:
!python lomo.py

## 5. (Optional) push results back to GitHub

Needs a GitHub token with repo write access, stored as a Colab secret named `GH_TOKEN`
(key icon in the left sidebar) -- never paste a token directly into a cell.

In [ ]:
from google.colab import userdata
import os
token = userdata.get('GH_TOKEN')
os.environ['GH_TOKEN'] = token
!cp -r /content/drive/MyDrive/cartcell/reports_lomo/* reports_lomo/
!git config user.email "colab@cartcell.local"
!git config user.name "Colab"
!git add reports_lomo
!git commit -m "LOMO progress from Colab"
!git push https://$GH_TOKEN@github.com/Ryan-Bae0121/Cartcell-Caspnet.git main